# 🦀 Day 1: clap v4 Deep Dive
---

Today we explore **clap** — Rust's most powerful CLI argument parser.

- **Derive API**: Use `#[derive(Parser)]` and `#[derive(Subcommand)]` for declarative CLI definitions
- **Arguments**: Positional, optional, flags with `#[arg(short, long, default_value)]`
- **Subcommands**: Build multi-command tools like `git` or `cargo`
- **Validation**: Value validators and custom error messages
- **Help**: Automatic help text generation

In [ ]:
:dep clap = { version = "4", features = ["derive"] }

In [ ]:
use clap::{Parser, Subcommand, ValueEnum};

/// A simple CLI with positional and optional args
#[derive(Parser)]
#[command(name = "greet", about = "A greeting program")]
struct GreetArgs {
    /// Name to greet
    name: String,
    /// Number of times to repeat
    #[arg(short, long, default_value = "1")]
    count: u32,
    /// Use uppercase
    #[arg(short, long)]
    shout: bool,
}

let args = GreetArgs::parse_from(["greet", "Rust", "-c", "3", "--shout"]);
let msg = if args.shout { args.name.to_uppercase() } else { args.name.clone() };
for _ in 0..args.count {
    println!("Hello, {}!", msg);
}

In [ ]:
/// File manager CLI with subcommands
#[derive(Parser)]
#[command(name = "fm", about = "Simple file manager CLI")]
struct FileManager {
    #[command(subcommand)]
    command: FileCommand,
}

#[derive(Subcommand)]
enum FileCommand {
    /// List files in a directory
    List {
        #[arg(default_value = ".")]
        path: String,
    },
    /// Copy a file
    Copy {
        source: String,
        dest: String,
    },
    /// Move a file
    Move {
        source: String,
        dest: String,
    },
    /// Delete a file
    Delete {
        path: String,
    },
}

let args = FileManager::parse_from(["fm", "list", "./src"]);
match &args.command {
    FileCommand::List { path } => println!("Would list: {}", path),
    FileCommand::Copy { source, dest } => println!("Would copy {} -> {}", source, dest),
    FileCommand::Move { source, dest } => println!("Would move {} -> {}", source, dest),
    FileCommand::Delete { path } => println!("Would delete: {}", path),
}

In [ ]:
/// Value validation with #[arg(value_parser)]
#[derive(Parser)]
#[command(name = "validate")]
struct ValidateArgs {
    #[arg(value_parser = clap::value_parser!(u32).range(1..=100))]
    port: u32,
}

let args = ValidateArgs::parse_from(["validate", "42"]);
println!("Port: {}", args.port);

## 📝 Exercise: Add a "search" subcommand

Add a `Search` variant to `FileCommand` that accepts:
- `pattern`: a regex pattern (String)
- `path`: optional path (default: ".")
- `--recursive` / `-r`: flag for recursive search

Parse and print the parsed args.

In [ ]:
// YOUR CODE HERE
// Add Search subcommand to FileCommand enum:
// Search {
//     pattern: String,
//     #[arg(default_value = ".")]
//     path: String,
//     #[arg(short, long)]
//     recursive: bool,
// }
// Then parse: fm search "\.rs$" --recursive

## 🎯 Key Takeaways

1. **clap** is the de facto standard for CLI parsing in Rust
2. Derive API with `#[derive(Parser)]` and `#[derive(Subcommand)]` keeps code declarative
3. `#[arg(short, long, default_value)]` controls argument behavior
4. Subcommands enable multi-command tools (like `git`, `cargo`)
5. Value validators catch invalid input early with clear error messages
6. Help text is auto-generated from doc comments

---
**Tomorrow:** Terminal UI with crossterm and ratatui